# Notebook 1: External validation of the SHAP-300 panel on METABRIC
**Paper:** SHAP-Guided Selection of Gene Panel with Repeated Nested Cross-Validation and PAM50-Exclusion Benchmarking for Interpretable Breast Cancer Subtype Classification from RNA-Seq, with External Validation on METABRIC
**Authors:** Neetu Ahlawat, Vishal Verma (Chaudhary Ranbir Singh University, Jind, India)

**What this notebook does (locked-panel external validation, Sections 3.7 and 4.7; Tables 12–13; Figs. 9–10):**
1. Rebuilds the TCGA-BRCA cohort used in the paper (514 tumours, four PAM50 subtypes).
2. Keeps only genes measured in both TCGA-BRCA and METABRIC (gene names only; METABRIC labels are never used for selection or training).
3. Selects the SHAP-300 panel and the comparator panels on all 514 TCGA tumours, then trains the four classifiers with the hyperparameters of Table 2.
4. Locks everything and predicts METABRIC once. METABRIC labels are used only for scoring.
5. Reports accuracy, balanced accuracy, macro/weighted F1 and weighted ROC-AUC with 95% bootstrap CIs, per-class metrics, confusion matrices, Holm-adjusted McNemar tests and a SHAP ranking comparison between cohorts.

**Data:** upload `HiSeqV2` (or `HiSeqV2.gz`) and the TCGA-BRCA clinical matrix from UCSC Xena (https://xenabrowser.net) to `/content/`. METABRIC (`brca_metabric.tar.gz`) is downloaded from cBioPortal automatically; if that fails, download it from https://www.cbioportal.org/study/summary?id=brca_metabric and upload it to `/content/`.

**Runtime (Colab CPU):** about 10–20 minutes.


In [ ]:
# ===== 1. Imports =====
import os, tarfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             precision_recall_fscore_support, confusion_matrix)
import xgboost as xgb
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
print("xgboost", xgb.__version__)

In [ ]:
# ===== 2. Config (change paths here if your files are elsewhere) =====
TCGA_EXPR    = "/content/HiSeqV2"              # UCSC Xena TCGA-BRCA gene expression, log2(RSEM+1); .gz also works
TCGA_CLIN    = "/content/BRCA_clinicalMatrix"  # UCSC Xena TCGA-BRCA clinical matrix
if not os.path.exists(TCGA_CLIN) and os.path.exists("/content/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix"):
    TCGA_CLIN = "/content/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix"   # name used by a manual Xena download
METABRIC_DIR = "/content/brca_metabric"        # cBioPortal METABRIC study folder
OUT_DIR      = "/content/metabric_validation"  # all tables and figures are saved here

SEED       = 42      # same seed as the paper
PANEL_SIZE = 300     # same panel size as the paper
N_BOOT     = 1000    # bootstrap resamples for 95% CIs
N_RANDOM   = 10      # number of Random-300 draws (negative control)

# Label encoding used in the paper: Basal-like = 0, HER2-enriched = 1, Luminal A = 2, Luminal B = 3
CLASSES     = ["Basal", "Her2", "LumA", "LumB"]
CLASS_NAMES = ["Basal-like", "HER2-enriched", "Luminal A", "Luminal B"]
LABEL_MAP   = {c: i for i, c in enumerate(CLASSES)}

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ===== 3. Download data if not present =====
def try_download(url, dest):
    try:
        print("Downloading", url)
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print("  Download failed:", e)
        return False

# TCGA (only if you have not uploaded your own copies)
XENA = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
if not (os.path.exists(TCGA_EXPR) or os.path.exists(TCGA_EXPR + ".gz")):
    if try_download(XENA + "TCGA.BRCA.sampleMap%2FHiSeqV2.gz", TCGA_EXPR + ".gz"):
        TCGA_EXPR = TCGA_EXPR + ".gz"
elif not os.path.exists(TCGA_EXPR):
    TCGA_EXPR = TCGA_EXPR + ".gz"
if not os.path.exists(TCGA_CLIN):
    try_download(XENA + "TCGA.BRCA.sampleMap%2FBRCA_clinicalMatrix", TCGA_CLIN)

# METABRIC from cBioPortal datahub
if not os.path.isdir(METABRIC_DIR):
    tgz = "/content/brca_metabric.tar.gz"
    if try_download("https://cbioportal-datahub.s3.amazonaws.com/brca_metabric.tar.gz", tgz):
        with tarfile.open(tgz) as t:
            t.extractall(os.path.dirname(METABRIC_DIR))

for p in [TCGA_EXPR, TCGA_CLIN, METABRIC_DIR]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

In [ ]:
# ===== 4. Gene-symbol harmonisation =====
# TCGA HiSeqV2 uses 2012-era HUGO symbols; METABRIC may use newer ones. Both datasets are mapped to the
# current symbol. Add more pairs here if Section 6 prints important unmatched genes.
ALIAS = {
    "GSG2": "HASPIN", "ORC6L": "ORC6", "KNTC2": "NDC80", "CDCA1": "NUF2",
    "KIAA0101": "PCLAF", "FAM64A": "PIMREG", "HN1": "JPT1", "KIAA1199": "CEMIP",
    "C1orf106": "INAVA", "MLF1IP": "CENPU", "CTSL2": "CTSV", "C17orf37": "MIEN1",
    "C10orf116": "ADIRF",
}
PAM50 = ["ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20","CDC6","CDH3","CENPF",
         "CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1","FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C",
         "KRT14","KRT17","KRT5","MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC","NAT1",
         "NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"]
assert len(PAM50) == 50

def harmonise(df):
    # df: samples x genes. Rename to current symbols and average duplicated symbols.
    df = df.rename(columns=lambda g: ALIAS.get(str(g).strip(), str(g).strip()))
    if df.columns.duplicated().any():
        df = df.T.groupby(level=0).mean().T
    return df

In [ ]:
# ===== 5. Load TCGA-BRCA exactly as in the paper =====
expr = pd.read_csv(TCGA_EXPR, sep="\t", index_col=0).T          # samples x genes
print("TCGA expression matrix:", expr.shape)
expr.index = expr.index.str[:15]
expr = expr[~expr.index.duplicated(keep="first")]
expr = harmonise(expr.apply(pd.to_numeric, errors="coerce"))
print("Example expression IDs:", list(expr.index[:3]))

# --- clinical file: detect separator and PAM50 column ---
print("\nClinical file used:", TCGA_CLIN)
clin = pd.read_csv(TCGA_CLIN, sep="\t", low_memory=False)
if clin.shape[1] == 1:                                   # file is comma-separated
    clin = pd.read_csv(TCGA_CLIN, sep=",", low_memory=False)
print("Clinical table shape:", clin.shape)

id_col = next((c for c in ["sampleID", "sample", "Sample ID", "SAMPLE_ID"] if c in clin.columns), clin.columns[0])
pam_col = next((c for c in ["PAM50_mRNA_nature2012", "PAM50Call_RNAseq", "PAM50"] if c in clin.columns), None)
print("ID column:", id_col, "| PAM50 column:", pam_col)
assert pam_col is not None, ("No PAM50 column found. This is probably not the TCGA BRCA_clinicalMatrix file. "
                             "Columns are: " + ", ".join(map(str, clin.columns[:30])))

clin = clin.set_index(id_col)
clin.index = clin.index.astype(str).str[:15]
clin = clin[~clin.index.duplicated(keep="first")]
print("Example clinical IDs:", list(clin.index[:3]))
print("PAM50 values in file:", clin[pam_col].value_counts(dropna=False).to_dict())

NAME_FIX = {"Luminal A": "LumA", "Luminal B": "LumB", "Basal-like": "Basal", "HER2-enriched": "Her2",
            "Normal-like": "Normal", "LumA": "LumA", "LumB": "LumB", "Basal": "Basal", "Her2": "Her2"}
lab_t = clin[pam_col].dropna().astype(str).str.strip().map(NAME_FIX)
lab_t = lab_t[lab_t.isin(CLASSES)]
ids_t = expr.index.intersection(lab_t.index)
assert len(ids_t) > 0, ("0 tumours matched between expression and clinical IDs. "
                        "Compare the example IDs printed above.")
X_tcga = expr.loc[ids_t]
y_tcga = lab_t.loc[ids_t].map(LABEL_MAP).astype(int).values
print("\nTCGA tumours with expression + four-class PAM50 label:", len(ids_t), "(paper: 514)")
print("Distinct patients:", pd.Index(ids_t).str[:12].nunique())
print(pd.Series(y_tcga).map(dict(enumerate(CLASS_NAMES))).value_counts())
if X_tcga.isna().any().any():
    X_tcga = X_tcga.fillna(X_tcga.median())

In [ ]:
# ===== 6. Load METABRIC (cBioPortal) =====
import glob, shutil, tarfile

# 6a. Extract the downloaded cBioPortal archive (works for "brca_metabric.tar.gz" or "brca_metabric (1).tar.gz")
tars = sorted(glob.glob("/content/brca_metabric*.tar.gz"), key=os.path.getsize, reverse=True)
assert tars, "No brca_metabric*.tar.gz found in /content - please upload it first"
tar_path = tars[0]
print("Archive:", tar_path, "-", round(os.path.getsize(tar_path) / 1e6, 1), "MB")

shutil.rmtree(METABRIC_DIR, ignore_errors=True)        # remove old/incomplete files
with tarfile.open(tar_path) as t:
    t.extractall("/content")                            # creates /content/brca_metabric/

mb_file = os.path.join(METABRIC_DIR, "data_mrna_illumina_microarray.txt")
assert os.path.exists(mb_file), "Expression file not found after extraction"
print("Using", mb_file, "-", round(os.path.getsize(mb_file) / 1e6, 1), "MB")

# 6b. Expression matrix (log2 Illumina intensities, NOT the z-score file)
mb = pd.read_csv(mb_file, sep="\t", comment="#", low_memory=False)
mb = mb.dropna(subset=["Hugo_Symbol"]).drop(columns=[c for c in ["Entrez_Gene_Id"] if c in mb.columns])
mb = mb.set_index("Hugo_Symbol").T                                  # samples x genes
mb = harmonise(mb.apply(pd.to_numeric, errors="coerce"))
print("METABRIC expression matrix:", mb.shape)

# 6c. PAM50 labels from the clinical files
pat = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_patient.txt"), sep="\t", comment="#", index_col="PATIENT_ID")
smp = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_sample.txt"), sep="\t", comment="#")
s2p = dict(zip(smp["SAMPLE_ID"], smp["PATIENT_ID"]))
lab_m_all = pd.Series({s: pat.loc[s2p[s], "CLAUDIN_SUBTYPE"] if s2p.get(s) in pat.index else np.nan
                       for s in mb.index})
print("METABRIC subtype labels (all, before filtering):")
print(lab_m_all.value_counts(dropna=False))

lab_m = lab_m_all[lab_m_all.isin(CLASSES)]       # drop claudin-low, Normal, NC and missing
X_mb = mb.loc[lab_m.index]
y_mb = lab_m.map(LABEL_MAP).astype(int).values
print("\nMETABRIC tumours kept (four classes):", len(y_mb))
mb_counts = pd.Series(y_mb).map(dict(enumerate(CLASS_NAMES))).value_counts()
print(mb_counts)

# 6d. Missing values: per-gene median within METABRIC (label-free); genes with >20% missing are dropped
miss = X_mb.isna().mean()
X_mb = X_mb.loc[:, miss <= 0.20]
X_mb = X_mb.fillna(X_mb.median())
print("Genes with >20% missing dropped:", int((miss > 0.20).sum()))

In [ ]:
# ===== 7. Common genes and per-cohort z-scoring =====
common = sorted(set(X_tcga.columns) & set(X_mb.columns))
pam50_common = [g for g in PAM50 if g in common]
print(f"TCGA genes: {X_tcga.shape[1]} | METABRIC genes: {X_mb.shape[1]} | common: {len(common)}")
print(f"PAM50 genes present in both: {len(pam50_common)}/50  missing: {sorted(set(PAM50) - set(pam50_common))}")

# Each cohort is standardised on itself (per gene, zero mean, unit variance). This removes the scale difference
# between RNA-Seq log2(RSEM+1) and Illumina log2 intensities. METABRIC labels are not used in this step.
Xt_all = pd.DataFrame(StandardScaler().fit_transform(X_tcga.values), index=X_tcga.index, columns=X_tcga.columns)
Xt = Xt_all[common]
Xm = pd.DataFrame(StandardScaler().fit_transform(X_mb[common].values), index=X_mb.index, columns=common)

In [ ]:
# ===== 8. Helper functions =====
def make_models(seed=SEED):
    # Same settings as Table 2 of the paper
    return {
        "XGBoost": XGBClassifier(objective="multi:softprob", n_estimators=300, max_depth=5,
                                 learning_rate=0.05, eval_metric="mlogloss", random_state=seed, n_jobs=-1),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=seed),
        "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=5000),
    }

def tree_shap(model, X):
    # Exact TreeSHAP (pred_contribs). Returns array (n_samples, n_classes, n_features)
    c = np.asarray(model.get_booster().predict(xgb.DMatrix(np.asarray(X)), pred_contribs=True))
    if c.ndim == 2:
        c = c.reshape(X.shape[0], -1, X.shape[1] + 1)
    return c[:, :, :-1]

def shap_select(X, y, k=PANEL_SIZE, seed=SEED):
    booster = make_models(seed)["XGBoost"].fit(X.values, y)
    imp = np.abs(tree_shap(booster, X.values)).mean(axis=(0, 1))
    order = np.argsort(-imp, kind="stable")
    n_nonzero = int((imp > 0).sum())
    if n_nonzero < k:
        print(f"  Warning: only {n_nonzero} genes have non-zero SHAP importance")
    return list(X.columns[order[:k]]), pd.Series(imp, index=X.columns).sort_values(ascending=False)

def fast_scores(y, p, n_cls=4):
    cm = np.bincount(y * n_cls + p, minlength=n_cls * n_cls).reshape(n_cls, n_cls)
    tp = np.diag(cm).astype(float); support = cm.sum(1); predicted = cm.sum(0)
    rec = np.divide(tp, support, out=np.zeros(n_cls), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(n_cls), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n_cls), where=(prec + rec) > 0)
    present = support > 0
    return tp.sum() / len(y), rec[present].mean(), f1[present].mean()

def bootstrap_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y)
    b = np.array([fast_scores(y[i], p[i]) for i in (rng.integers(0, n, n) for _ in range(n_boot))])
    lo, hi = np.percentile(b, [2.5, 97.5], axis=0) * 100
    return lo, hi    # order: accuracy, balanced accuracy, macro F1

def evaluate(y, pred, proba):
    return {
        "Accuracy (%)": accuracy_score(y, pred) * 100,
        "Balanced accuracy (%)": balanced_accuracy_score(y, pred) * 100,
        "Macro F1 (%)": f1_score(y, pred, average="macro") * 100,
        "Weighted F1 (%)": f1_score(y, pred, average="weighted") * 100,
        "Weighted ROC-AUC (%)": roc_auc_score(y, proba, multi_class="ovr", average="weighted", labels=[0, 1, 2, 3]) * 100,
    }

def mcnemar_exact(y, p1, p2):
    a = (p1 == y) & (p2 != y); b = (p1 != y) & (p2 == y)
    n = int(a.sum() + b.sum())
    return 1.0 if n == 0 else binomtest(int(a.sum()), n, 0.5).pvalue

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

In [ ]:
# ===== 9. Build all panels on the 514 TCGA tumours (METABRIC is not touched) =====
import time, json

RUN_FULL_GENE_PANEL = True   # False = skip the paper-style (all 20,530 genes) panel and save ~half the time

def shap_select_verbose(X, y, label, k=PANEL_SIZE, seed=SEED):
    cache = f"{OUT_DIR}/panel_{label}.json"
    if os.path.exists(cache):                                   # reuse a finished panel after an interruption
        print(f"[{label}] loaded saved panel from {cache}")
        return json.load(open(cache))
    t0 = time.time()
    print(f"[{label}] fitting XGBoost on {X.shape[0]} tumours x {X.shape[1]} genes "
          f"(300 rounds, progress every 25) ...")
    booster = make_models(seed)["XGBoost"]
    booster.fit(X.values, y, eval_set=[(X.values, y)], verbose=25)
    print(f"[{label}] fit done in {(time.time() - t0) / 60:.1f} min, computing TreeSHAP ...")
    imp = np.abs(tree_shap(booster, X.values)).mean(axis=(0, 1))
    order = np.argsort(-imp, kind="stable")
    if (imp > 0).sum() < k:
        print(f"  Warning: only {(imp > 0).sum()} genes have non-zero SHAP importance")
    panel = list(X.columns[order[:k]])
    json.dump(panel, open(cache, "w"))
    print(f"[{label}] done in {(time.time() - t0) / 60:.1f} min total; panel saved to {cache}\n")
    return panel

# 9a. Primary panel: SHAP-300 selected only from genes measured in both cohorts
shap_panel = shap_select_verbose(Xt, y_tcga, "shap300_common")

# 9b. Paper-style panel: SHAP-300 selected from all TCGA genes (to report how many genes METABRIC measures)
if RUN_FULL_GENE_PANEL:
    shap_panel_full = shap_select_verbose(Xt_all, y_tcga, "shap300_allgenes")
    full_matched = [g for g in shap_panel_full if g in common]
    print(f"Paper-style panel: {len(full_matched)}/{PANEL_SIZE} genes present in METABRIC")
    print("Unmatched genes:", [g for g in shap_panel_full if g not in common])
    print(f"Overlap between primary and paper-style SHAP-300: {len(set(shap_panel) & set(shap_panel_full))}/{PANEL_SIZE}")

# 9c. ANOVA-300 comparator (fast)
F, _ = f_classif(Xt.values, y_tcga)
anova_panel = list(Xt.columns[np.argsort(-np.nan_to_num(F), kind="stable")[:PANEL_SIZE]])
print(f"PAM50 genes inside primary SHAP-300: {len(set(shap_panel) & set(PAM50))}")

panels = {"SHAP-300": shap_panel}
if RUN_FULL_GENE_PANEL:
    panels["SHAP-300 (all-gene selection, matched genes)"] = full_matched
panels["ANOVA-300"] = anova_panel
panels[f"PAM50 ({len(pam50_common)} genes)"] = pam50_common
panels[f"All common genes ({len(common)})"] = common

pd.Series(shap_panel, name="gene").to_csv(f"{OUT_DIR}/shap300_panel_common_genes.csv", index=False)
print("\nPanels ready:", {k: len(v) for k, v in panels.items()})

In [ ]:
# ===== 10. Train on TCGA, predict METABRIC once =====
rows, preds, probas, fitted = [], {}, {}, {}
ci_names = ["Accuracy (%)", "Balanced accuracy (%)", "Macro F1 (%)"]
for pname, genes in panels.items():
    for mname, model in make_models().items():
        model.fit(Xt[genes].values, y_tcga)
        pr = model.predict_proba(Xm[genes].values); pd_ = pr.argmax(1)
        preds[(pname, mname)], probas[(pname, mname)], fitted[(pname, mname)] = pd_, pr, model
        r = {"Feature set": pname, "Model": mname, "n genes": len(genes), **evaluate(y_mb, pd_, pr)}
        lo, hi = bootstrap_ci(y_mb, pd_)
        for j, c in enumerate(ci_names):
            r[c.replace(" (%)", " 95% CI")] = f"{lo[j]:.2f}-{hi[j]:.2f}"
        rows.append(r)
        print(f"{pname:45s} {mname:20s} acc = {r['Accuracy (%)']:.2f}%")

# Random-300 negative control: N_RANDOM draws, mean +/- SD
rng = np.random.default_rng(SEED); rand_res = {m: [] for m in make_models()}
for d in range(N_RANDOM):
    genes = list(rng.choice(common, PANEL_SIZE, replace=False))
    for mname, model in make_models(SEED + d).items():
        model.fit(Xt[genes].values, y_tcga); pr = model.predict_proba(Xm[genes].values)
        rand_res[mname].append(evaluate(y_mb, pr.argmax(1), pr))
for mname, lst in rand_res.items():
    dfr = pd.DataFrame(lst)
    r = {"Feature set": f"Random-300 (mean ± SD of {N_RANDOM})", "Model": mname, "n genes": PANEL_SIZE}
    for c in dfr.columns:
        r[c] = dfr[c].mean(); r[c.replace(" (%)", " SD")] = dfr[c].std()
    rows.append(r)

res = pd.DataFrame(rows)
majority = mb_counts.max() / mb_counts.sum() * 100
print(f"\nMajority-class (Luminal A) base rate in METABRIC: {majority:.2f}%")
show = ["Feature set", "Model", "n genes", "Accuracy (%)", "Accuracy 95% CI", "Balanced accuracy (%)",
        "Macro F1 (%)", "Weighted F1 (%)", "Weighted ROC-AUC (%)"]
display(res[show].round(2))

In [ ]:
# ===== 11. Per-class metrics and McNemar tests for SHAP-300 =====
pc_rows = []
for mname in make_models():
    p, r, f, s = precision_recall_fscore_support(y_mb, preds[("SHAP-300", mname)], labels=[0, 1, 2, 3], zero_division=0)
    pc_rows.append({"Model": mname, **{f"{CLASS_NAMES[i]} (P / R / F1)": f"{p[i]:.2f} / {r[i]:.2f} / {f[i]:.2f}" for i in range(4)}})
per_class = pd.DataFrame(pc_rows); display(per_class)

# Paired comparison on the same METABRIC tumours: exact McNemar test, Holm-adjusted within each classifier
mc_rows = []
others = [p for p in panels if p != "SHAP-300"]
for mname in make_models():
    base = preds[("SHAP-300", mname)]
    raw = [mcnemar_exact(y_mb, base, preds[(o, mname)]) for o in others]
    adj = holm(raw)
    for o, pr_, pa in zip(others, raw, adj):
        diff = (accuracy_score(y_mb, base) - accuracy_score(y_mb, preds[(o, mname)])) * 100
        mc_rows.append({"Model": mname, "SHAP-300 minus": o, "Accuracy difference (points)": round(diff, 2),
                        "McNemar p": round(pr_, 4), "Holm-adjusted p": round(pa, 4)})
mcn = pd.DataFrame(mc_rows); display(mcn)

In [ ]:
# ===== 12. Figures =====
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (letter, mname) in zip(axes.ravel(), zip("abcd", make_models())):
    cm = confusion_matrix(y_mb, preds[("SHAP-300", mname)], labels=[0, 1, 2, 3])
    ax.imshow(cm, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted subtype"); ax.set_ylabel("True subtype"); ax.set_title(f"{letter}  {mname}")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_metabric_confusion_shap300.png", dpi=600); plt.show()

plot_sets = ["SHAP-300", "ANOVA-300", f"PAM50 ({len(pam50_common)} genes)", f"All common genes ({len(common)})"]
models = list(make_models()); w = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
for k, ps in enumerate(plot_sets):
    sub = res[res["Feature set"] == ps].set_index("Model").loc[models]
    lo = sub["Accuracy 95% CI"].str.split("-").str[0].astype(float)
    hi = sub["Accuracy 95% CI"].str.split("-").str[1].astype(float)
    acc = sub["Accuracy (%)"]
    ax.bar(np.arange(4) + (k - 1.5) * w, acc, w, yerr=[acc - lo, hi - acc], capsize=3, label=ps)
ax.axhline(majority, ls=":", color="grey", label="Majority-class rate")
ax.set_xticks(range(4)); ax.set_xticklabels(models); ax.set_ylabel("METABRIC accuracy (%)")
ax.set_ylim(0, 100); ax.legend(fontsize=8, loc="lower right")
ax.set_title("External validation on METABRIC (models trained on TCGA-BRCA only); bars = 95% bootstrap CI")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_metabric_accuracy_by_feature_set.png", dpi=600); plt.show()

In [ ]:
# ===== 13. SHAP ranking on METABRIC vs TCGA =====
xgb_final = fitted[("SHAP-300", "XGBoost")]
imp_m = pd.Series(np.abs(tree_shap(xgb_final, Xm[shap_panel].values)).mean(axis=(0, 1)), index=shap_panel)
imp_t = pd.Series(np.abs(tree_shap(xgb_final, Xt[shap_panel].values)).mean(axis=(0, 1)), index=shap_panel)
rho, p_rho = spearmanr(imp_t, imp_m)
top_t, top_m = list(imp_t.sort_values(ascending=False).index[:20]), list(imp_m.sort_values(ascending=False).index[:20])
print(f"Spearman correlation of gene importance (300 genes), TCGA vs METABRIC: rho = {rho:.3f}, p = {p_rho:.2e}")
print(f"Top-20 genes shared: {len(set(top_t) & set(top_m))}/20")
shap_tab = pd.DataFrame({"Rank": range(1, 21), "TCGA gene": top_t, "TCGA mean |SHAP|": imp_t[top_t].round(4).values,
                         "METABRIC gene": top_m, "METABRIC mean |SHAP|": imp_m[top_m].round(4).values})
display(shap_tab)

In [ ]:
# ===== 14. Save everything =====
with pd.ExcelWriter(f"{OUT_DIR}/metabric_external_validation.xlsx") as w:
    pd.DataFrame({"Item": ["TCGA tumours", "METABRIC tumours (4 classes)", "Common genes", "PAM50 genes in both",
                           "Paper-style SHAP-300 genes present in METABRIC", "Majority-class rate METABRIC (%)"],
                  "Value": [len(y_tcga), len(y_mb), len(common), len(pam50_common), len(full_matched), round(majority, 2)]}
                 ).to_excel(w, sheet_name="Cohort summary", index=False)
    mb_counts.rename("METABRIC count").to_frame().to_excel(w, sheet_name="METABRIC classes")
    res.round(2).to_excel(w, sheet_name="Overall metrics", index=False)
    per_class.to_excel(w, sheet_name="Per-class SHAP-300", index=False)
    mcn.to_excel(w, sheet_name="McNemar tests", index=False)
    shap_tab.to_excel(w, sheet_name="SHAP top-20", index=False)
res.round(2).to_csv(f"{OUT_DIR}/metabric_overall_metrics.csv", index=False)
print("Saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))